# 👥 Collaborative Filtering – Furniture Shop ML Service

Notebook này trình bày phương pháp **Collaborative Filtering** được triển khai trong `ml_service/app/services/recommendation_service.py`.

## Tổng quan
Hệ thống đề xuất dựa trên **hành vi của những người dùng tương tự** — không cần biết nội dung sản phẩm, chỉ cần biết ai mua/đánh giá gì.

### Nguyên lý:
> *"Nếu user A và user B đều mua sản phẩm X, và user B còn mua thêm Y, thì user A nhiều khả năng cũng thích Y."*

| Bước | Tên | Mô tả |
|------|-----|-------|
| 1 | Build Matrix | Tạo ma trận User-Item từ interactions |
| 2 | User Similarity | Tính cosine similarity giữa users |
| 3 | Weighted Prediction | Dự đoán điểm SP chưa mua bằng weighted average |
| 4 | Fallback | Nếu data mỏng → xếp theo tổng score toàn hệ thống |

## 1. Thiết lập môi trường & Nạp dữ liệu

> ⚠️ **Lưu ý**: Chạy cell này trước tất cả các cell khác — đây là cell duy nhất cần chạy để khởi tạo tất cả biến.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

# --- Nạp dữ liệu ---
reviews_raw  = pd.read_csv('furniture_shop.reviews.csv')
orders_raw   = pd.read_csv('furniture_shop.orders.csv')
products_raw = pd.read_csv('furniture_shop.products.csv')

# Map product_id -> name (dùng xuyên suốt notebook)
product_names = dict(zip(products_raw['_id'].astype(str), products_raw['name']))

print(f'✅ Thư viện đã sẵn sàng')
print(f'⭐ Reviews : {len(reviews_raw)}')
print(f'🛒 Orders  : {len(orders_raw)}')
print(f'📦 Products: {len(products_raw)}')

## 2. Xây dựng User-Item Interaction Matrix → `interactions_to_df()`

**Công thức tính điểm tương tác:**
```
score(user, product) = Σ(rating từ reviews) + Σ(quantity × 1.5 từ orders)
```
Mua hàng được trọng số **×1.5** vì phản ánh hành vi mạnh hơn đánh giá.

In [ ]:
def build_interactions_df(reviews_raw, orders_raw):
    """Tái hiện interactions_to_df() từ data_prep_service.py"""
    signals = defaultdict(float)

    # Từ reviews: score += rating
    for _, r in reviews_raw.iterrows():
        uid = str(r['user'])
        pid = str(r['product'])
        if uid and pid:
            signals[(uid, pid)] += float(r.get('rating', 0) or 0)

    # Từ orders: score += quantity * 1.5
    item_p_cols = [c for c in orders_raw.columns if c.startswith('items[') and c.endswith('.product')]
    item_q_cols = [c for c in orders_raw.columns if c.startswith('items[') and c.endswith('.quantity')]

    for _, o in orders_raw.iterrows():
        uid = str(o['user'])
        for pc, qc in zip(item_p_cols, item_q_cols):
            pid = o.get(pc)
            qty = o.get(qc)
            if pd.notna(pid) and pd.notna(qty):
                signals[(uid, str(pid))] += float(qty) * 1.5

    rows = [{'user_id': u, 'product_id': p, 'score': s}
            for (u, p), s in signals.items()]
    return pd.DataFrame(rows)


interactions = build_interactions_df(reviews_raw, orders_raw)
print(f'✅ Bảng tương tác: {interactions.shape}')
print(f'   Số users   : {interactions["user_id"].nunique()}')
print(f'   Số products: {interactions["product_id"].nunique()}')

# --- Visualise phân phối ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

interactions['score'].hist(bins=20, ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Phân phối Điểm Tương tác (score)', fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Tần suất')

user_counts = interactions.groupby('user_id').size().sort_values(ascending=False)
axes[1].bar(range(len(user_counts)), user_counts.values, color='#2ecc71', edgecolor='white')
axes[1].set_title('Số Sản Phẩm Đã Tương Tác mỗi User', fontweight='bold')
axes[1].set_xlabel('User (sort by count)')
axes[1].set_ylabel('Số sản phẩm')

plt.tight_layout()
plt.show()
interactions.head(8)

## 3. Pivot Table → User-Item Matrix

Ma trận có kích thước **[số_user × số_product]**, giá trị là interaction score (0 = chưa tương tác).

In [ ]:
# Tái hiện logic trong collaborative_recommendation()
matrix = interactions.pivot_table(
    index      = 'user_id',
    columns    = 'product_id',
    values     = 'score',
    fill_value = 0
)

print(f'📊 User-Item Matrix: {matrix.shape}')
print(f'   {matrix.shape[0]} users × {matrix.shape[1]} products')
print(f'   Sparsity: {(matrix == 0).sum().sum() / matrix.size * 100:.1f}% (giá trị = 0)')

# Visualise heatmap
matrix_display = matrix.copy()
matrix_display.columns = [product_names.get(str(c), str(c))[:15] for c in matrix_display.columns]
matrix_display.index   = [f'User...{str(i)[-6:]}' for i in matrix_display.index]

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(matrix_display, cmap='Blues', ax=ax, linewidths=0.3,
            cbar_kws={'label': 'Interaction Score'})
ax.set_title('User-Item Interaction Matrix', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 4. User-User Similarity

**Cosine Similarity** giữa các hàng trong User-Item Matrix:
$$\text{sim}(u, v) = \frac{\vec{u} \cdot \vec{v}}{|\vec{u}||\vec{v}|}$$

In [ ]:
user_sim = cosine_similarity(matrix)
sim_df   = pd.DataFrame(user_sim, index=matrix.index, columns=matrix.index)

print(f'📐 User Similarity Matrix: {sim_df.shape}')

# Rút gọn index để hiển thị
display_index       = [f'...{str(i)[-6:]}' for i in sim_df.index]
sim_display         = sim_df.copy()
sim_display.index   = display_index
sim_display.columns = display_index

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sim_display, annot=True, fmt='.3f', cmap='YlGn',
            vmin=0, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Cosine Similarity'})
ax.set_title('Ma trận User-User Similarity', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Dự đoán điểm & Đề xuất

**Công thức Weighted Average:**
$$\hat{r}(u, p) = \frac{\sum_{v \neq u} \text{sim}(u,v) \cdot r(v,p)}{\sum_{v \neq u} |\text{sim}(u,v)|}$$

In [ ]:
def collaborative_recommend(target_user_id: str, top_k: int = 5):
    """
    Tái hiện collaborative_recommendation() từ recommendation_service.py

    Công thức dự đoán:
        pred(u, p) = Σ_v [sim(u,v) * rating(v,p)] / Σ_v |sim(u,v)|
    """
    if target_user_id not in matrix.index:
        print(f'⚠️ User không tồn tại trong matrix → fallback')
        product_scores = interactions.groupby('product_id')['score'].sum().sort_values(ascending=False)
        return [{'product_id': pid, 'name': product_names.get(str(pid), ''), 'score': float(s)}
                for pid, s in product_scores.head(top_k).items()]

    # Lấy weights = similarity của target user với tất cả users khác
    weights = sim_df.loc[target_user_id].drop(target_user_id)

    if weights.abs().sum() == 0:
        print('⚠️ Weights rỗng (user không có điểm tương đồng) → fallback')
        product_scores = interactions.groupby('product_id')['score'].sum().sort_values(ascending=False)
        return [{'product_id': pid, 'name': product_names.get(str(pid), ''), 'score': float(s)}
                for pid, s in product_scores.head(top_k).items()]

    # Weighted average rating
    weighted = np.dot(weights.values, matrix.loc[weights.index].values)
    pred     = weighted / (np.abs(weights.values).sum() + 1e-8)

    # Loại sản phẩm user đã mua/đánh giá
    target_seen = set(matrix.columns[matrix.loc[target_user_id] > 0])
    ranked_idx  = np.argsort(pred)[::-1]
    columns     = matrix.columns.tolist()

    results = []
    for idx in ranked_idx:
        pid = columns[idx]
        if pid in target_seen:
            continue
        results.append({
            'product_id': pid,
            'name'      : product_names.get(str(pid), str(pid)[:12]),
            'score'     : round(float(pred[idx]), 4)
        })
        if len(results) >= top_k:
            break
    return results


# Test với user đầu tiên
target_user = matrix.index[0]
print(f'👤 Đề xuất Collaborative Filtering cho user: ...{str(target_user)[-12:]}')
print(f'📋 Sản phẩm user đã tương tác: {(matrix.loc[target_user] > 0).sum()} sản phẩm\n')

recs    = collaborative_recommend(target_user, top_k=5)
recs_df = pd.DataFrame(recs)
recs_df

In [ ]:
# Visualise kết quả
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1: Recommended products score
if not recs_df.empty and 'name' in recs_df.columns and 'score' in recs_df.columns:
    n     = len(recs_df)
    # Tạo danh sách màu xanh lá từ nhạt → đậm
    greens = sns.color_palette('Greens', n + 2)[2:]  # bỏ 2 màu đầu quá nhạt
    axes[0].barh(
        recs_df['name'].str[:28],
        recs_df['score'],
        color=list(reversed(greens))
    )
    axes[0].set_title(f'Top đề xuất Collaborative cho\nUser ...{str(target_user)[-12:]}',
                      fontweight='bold')
    axes[0].set_xlabel('Predicted Score')
else:
    axes[0].text(0.5, 0.5, 'Không có đề xuất', ha='center', va='center', fontsize=12)
    axes[0].set_title('Top đề xuất Collaborative', fontweight='bold')

# Chart 2: Weights (similarity với target user)
weights   = sim_df.loc[target_user].drop(target_user)
short_ids = [f'...{str(i)[-6:]}' for i in weights.index]
bar_colors = [
    '#e74c3c' if w > 0.5 else '#3498db' if w > 0.2 else '#95a5a6'
    for w in weights.values
]
axes[1].bar(short_ids, weights.values, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[1].axhline(y=0.5, color='red',  linestyle='--', alpha=0.6, label='High (>0.5)')
axes[1].axhline(y=0.2, color='blue', linestyle='--', alpha=0.6, label='Low (>0.2)')
axes[1].set_title('Trọng số Similarity với các User khác', fontweight='bold')
axes[1].set_ylabel('Cosine Similarity')
axes[1].set_xlabel('User ID (rút gọn)')
axes[1].legend(fontsize=8)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 6. Chạy đề xuất cho tất cả Users

In [ ]:
# Đề xuất cho từng user trong hệ thống
print('🔍 Kết quả đề xuất cho tất cả users (top-5):')
print('=' * 65)

for uid in matrix.index:
    recs_i = collaborative_recommend(str(uid), top_k=5)
    seen_count = int((matrix.loc[uid] > 0).sum())
    print(f'\n👤 User ...{str(uid)[-12:]} (đã tương tác {seen_count} SP):')
    if recs_i:
        for r in recs_i:
            print(f"   • {r['name'][:35]:<37} score={r['score']:.4f}")
    else:
        print('   (Không có đề xuất)')

## 7. Phân tích mức độ tương đồng giữa users

In [ ]:
# So sánh hành vi tổng hợp của tất cả users
user_stats = []
for uid in matrix.index:
    row = matrix.loc[uid]
    user_stats.append({
        'user_id'    : f'...{str(uid)[-8:]}',
        'n_products' : int((row > 0).sum()),
        'total_score': float(row.sum()),
        'avg_score'  : float(row[row > 0].mean()) if (row > 0).any() else 0.0,
    })

user_stats_df = pd.DataFrame(user_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

palette1 = sns.color_palette('Set2', len(user_stats_df))
bars = axes[0].bar(user_stats_df['user_id'], user_stats_df['n_products'], color=palette1)
axes[0].bar_label(bars, padding=2)
axes[0].set_title('Số Sản Phẩm Đã Tương Tác theo User', fontweight='bold')
axes[0].set_ylabel('Số sản phẩm')
axes[0].tick_params(axis='x', rotation=20)

palette2 = sns.color_palette('Set3', len(user_stats_df))
bars2 = axes[1].bar(user_stats_df['user_id'], user_stats_df['total_score'], color=palette2)
axes[1].bar_label(bars2, fmt='%.0f', padding=2)
axes[1].set_title('Tổng Điểm Tương Tác theo User', fontweight='bold')
axes[1].set_ylabel('Tổng score')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()
user_stats_df

## 8. Predicted Score heatmap – tất cả users × products chưa tương tác

In [ ]:
# Tính pred score cho mỗi user × mỗi product (bao gồm cả đã thấy)
pred_rows = {}
for uid in matrix.index:
    weights_i = sim_df.loc[uid].drop(uid)
    if weights_i.abs().sum() == 0:
        pred_rows[str(uid)] = np.zeros(len(matrix.columns))
        continue
    w = np.dot(weights_i.values, matrix.loc[weights_i.index].values)
    pred_rows[str(uid)] = w / (np.abs(weights_i.values).sum() + 1e-8)

pred_df = pd.DataFrame(pred_rows, index=matrix.columns).T

# Rút gọn tên
pred_display         = pred_df.copy()
pred_display.columns = [product_names.get(str(c), str(c))[:14] for c in pred_display.columns]
pred_display.index   = [f'...{str(i)[-6:]}' for i in pred_display.index]

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pred_display, annot=True, fmt='.1f', cmap='RdYlGn',
            ax=ax, linewidths=0.4, cbar_kws={'label': 'Predicted Score'})
ax.set_title('Predicted Scores – mỗi User × mỗi Product\n(chỉ những SP chưa tương tác sẽ được đề xuất)', fontsize=12, fontweight='bold')
plt.xticks(rotation=40, ha='right', fontsize=7)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## 9. Tóm tắt luồng xử lý

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║         Collaborative Filtering – Luồng Xử Lý                   ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. INPUT: reviews[] + orders[]  →  interactions_to_df()         ║
║      score = rating(review) + quantity × 1.5(order)              ║
║                                                                  ║
║  2. pivot_table(user_id, product_id, score, fill_value=0)        ║
║     → User-Item Matrix [U × P]                                   ║
║                                                                  ║
║  3. cosine_similarity(matrix) → User-User Sim [U × U]            ║
║                                                                  ║
║  4. weights = sim_df.loc[target].drop(target)                    ║
║     if weights.abs().sum() == 0 → FALLBACK                       ║
║                                                                  ║
║  5. weighted = dot(weights, matrix[other_users])                 ║
║     pred = weighted / (sum(|weights|) + 1e-8)                   ║
║                                                                  ║
║  6. argsort(pred)[::-1] → bỏ qua seen → top_k                  ║
║                                                                  ║
║  7. Fallback: sum(score) per product (popularity)                ║
║                                                                  ║
║  8. OUTPUT: [{product_id, name, score}, ...]                     ║
╚══════════════════════════════════════════════════════════════════╝
""")